# Do Markets Correct Themselves?


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
RAW = "../data/raw"
PROC = "../data/processed"

np.random.seed(42)


## 1. Data preparation

I use monthly data for oil, wheat, and copper. The goal is to see how these markets behave after large price movements.


In [ ]:
files = {
    "wheat_price": "wheat_price.csv",
    "oil_price": "oil_price.csv",
    "oil_production": "oil_production.csv",
    "oil_stocks": "oil_stocks.csv",
    "oil_consumption": "oil_consumption.csv",
    "copper_price": "copper_price.csv",
    "cpi": "cpi.csv",
    "fedfunds": "fedfunds.csv"
}

raw_data = {
    name: pd.read_csv(f"{RAW}/{file}", parse_dates=["date"])
    for name, file in files.items()
}

summary = pd.DataFrame({
    name: {
        "observations": len(df),
        "start": df["date"].min().date(),
        "end": df["date"].max().date(),
        "missing": df.iloc[:, 1].isna().sum()
    }
    for name, df in raw_data.items()
}).T

summary

### Data cleaning

The processed datasets are loaded and checked for missing values.


In [ ]:
oil = pd.read_csv(f"{PROC}/oil_panel.csv", parse_dates=["date"], index_col="date")
wheatp = pd.read_csv(f"{PROC}/wheat_panel.csv", parse_dates=["date"], index_col="date")
copperp = pd.read_csv(f"{PROC}/copper_panel.csv", parse_dates=["date"], index_col="date")
combined = pd.read_csv(f"{PROC}/combined_prices.csv", parse_dates=["date"], index_col="date")

for name, df in [("Oil", oil), ("Wheat", wheatp), ("Copper", copperp)]:
    print(name, df.shape, df.index.min().date(), df.index.max().date())

oil.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))

for ax, (name, df) in zip(axes, [("Oil", oil), ("Wheat", wheatp), ("Copper", copperp)]):
    sns.heatmap(df.isna().T, cbar=False, ax=ax)
    ax.set_title(f"{name}: missing values")
    ax.set_xlabel("")
    ax.set_xticks([])

plt.tight_layout()
plt.show()

## 2. Exploratory data analysis

First, I look at the main price and market series before running the statistical models.


### 2.1 Summary statistics

These tables give a basic overview of the data.


In [ ]:
print("Oil market")
display(oil[["wti_price_usd_per_bbl", "oil_production_kbd", "oil_stocks_kbbl", "oil_consumption_kbbl"]].describe().round(1))

print("Wheat market")
display(wheatp[["wheat_price_usd_per_ton"]].describe().round(1))

print("Copper market")
display(copperp[["copper_price_usd_per_ton"]].describe().round(1))

### 2.2 Time series overview

The plots show how prices and related oil variables changed over time.


In [ ]:

fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
axes[0].plot(oil.index, oil["wti_price_usd_per_bbl"], color="#1f77b4")
axes[0].set_title("WTI crude oil price (nominal, USD/bbl)")
axes[1].plot(oil.index, oil["oil_production_kbd"], color="#2ca02c")
axes[1].set_title("U.S. crude oil production (thousand bbl/day)")
axes[2].plot(oil.index, oil["oil_stocks_kbbl"], color="#d62728")
axes[2].set_title("U.S. crude oil ending stocks, ex. SPR (thousand bbl)")
axes[3].plot(oil.index, oil["oil_consumption_kbbl"], color="#9467bd")
axes[3].set_title("U.S. petroleum product supplied (consumption proxy, thousand bbl)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(wheatp.index, wheatp["wheat_price_usd_per_ton"], label="Nominal", alpha=0.5)
axes[0].plot(wheatp.index, wheatp["wheat_price_real"], label="Real (Jan-2020 USD)", color="#8c564b")
axes[0].set_title("Global wheat price: nominal vs. real")
axes[0].legend()
axes[1].plot(copperp.index, copperp["copper_price_usd_per_ton"], label="Nominal", alpha=0.5)
axes[1].plot(copperp.index, copperp["copper_price_real"], label="Real (Jan-2020 USD)", color="#17becf")
axes[1].set_title("Global copper price: nominal vs. real")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 2.3 Rolling statistics

I use a 12-month rolling mean and volatility to see how the level and variability of prices change over time.


In [ ]:

fig, axes = plt.subplots(3, 2, figsize=(13, 9))
series_map = [
    ("Oil (real, $/bbl)", oil["wti_price_real"], axes[0]),
    ("Wheat (real, $/ton)", wheatp["wheat_price_real"], axes[1]),
    ("Copper (real, $/ton)", copperp["copper_price_real"], axes[2]),
]
for label, s, (ax_mean, ax_vol) in series_map:
    roll_mean = s.rolling(12).mean()
    ret = np.log(s).diff()
    roll_vol = ret.rolling(12).std() * np.sqrt(12) * 100  # annualized, %
    ax_mean.plot(s.index, s, alpha=0.35, color="gray", label="Monthly")
    ax_mean.plot(roll_mean.index, roll_mean, color="#1f77b4", label="12m rolling mean")
    ax_mean.set_title(f"{label}: level & 12m mean")
    ax_mean.legend(fontsize=8)
    ax_vol.plot(roll_vol.index, roll_vol, color="#d62728")
    ax_vol.set_title(f"{label}: 12m rolling annualized volatility (%)")
plt.tight_layout()
plt.show()


### 2.4 Price levels and monthly returns

I compare the distribution of price levels with monthly log-returns.


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
markets = [
    ("Oil", oil["wti_price_real"]),
    ("Wheat", wheatp["wheat_price_real"]),
    ("Copper", copperp["copper_price_real"]),
]
for i, (name, s) in enumerate(markets):
    sns.histplot(s.dropna(), kde=True, ax=axes[0, i], color="#1f77b4")
    axes[0, i].set_title(f"{name}: real price level")
    ret = np.log(s).diff().dropna() * 100
    sns.histplot(ret, kde=True, ax=axes[1, i], color="#d62728")
    axes[1, i].axvline(0, color="k", lw=1)
    axes[1, i].set_title(f"{name}: monthly log-return (%)\nskew={ret.skew():.2f}, kurtosis={ret.kurtosis():.2f}")
plt.tight_layout()
plt.show()


The return distributions are centered around zero, with some large movements.


### 2.5 Outlier detection

I flag unusually large monthly returns using a simple z-score rule.


In [ ]:
def flag_outliers(returns, z=3):
    z_scores = (returns - returns.mean()) / returns.std()
    return returns[z_scores.abs() > z]

for name, series in markets:
    returns = np.log(series).diff().dropna() * 100
    outliers = flag_outliers(returns)
    print(f"{name}: {len(outliers)} months with |z| > 3")
    print(outliers.round(1).to_string())

Some of the flagged observations occur around major market events. This is useful context, but the outlier rule is only a simple screening method.


### 2.6 Correlation analysis

I compare the main oil variables using correlations.


In [ ]:
oil_corr = oil[[
    "wti_price_usd_per_bbl", "oil_production_kbd",
    "oil_stocks_kbbl", "oil_consumption_kbbl", "cpi", "fedfunds"
]].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(oil_corr, annot=True, fmt=".2f", center=0, ax=ax)
ax.set_title("Oil market correlations")
plt.tight_layout()
plt.show()

Level correlations can be affected by common trends, so I use changes in the later analysis.


## 3. Statistical modeling

The next steps use simple time-series regressions and tests.


In [ ]:
d = oil.dropna(subset=["wti_price_usd_per_bbl", "oil_production_kbd"]).copy()
d["log_price"] = np.log(d["wti_price_usd_per_bbl"])
d["log_prod"] = np.log(d["oil_production_kbd"])

adf_results = {}
for col in ["log_price", "log_prod"]:
    stat, p, *_ = adfuller(d[col])
    adf_results[col] = (stat, p)

for col, (stat, p) in adf_results.items():
    print(f"{col}: ADF={stat:.3f}, p={p:.4f}")

The ADF test checks whether the series appear to be stationary.


In [ ]:

d["dlog_price"] = d["log_price"].diff()
d["dlog_prod"]  = d["log_prod"].diff()
dd = d.dropna(subset=["dlog_price", "dlog_prod"])

for col in ["dlog_price", "dlog_prod"]:
    stat, p, *_ = adfuller(dd[col])
    print(f"  {col:12s}  ADF stat={stat:.3f}   p-value={p:.4g}   (stationary: {p < 0.05})")


In [ ]:
# Levels model, shown as a comparison
a = sm.add_constant(d["log_prod"])
model_levels = sm.OLS(d["log_price"], a).fit()

# Growth-rate model
X = sm.add_constant(dd["dlog_prod"])
model_diff = sm.OLS(
    dd["dlog_price"], X
).fit(cov_type="HAC", cov_kwds={"maxlags": 6})

print("Levels model")
print(model_levels.summary())

print("\nGrowth-rate model")
print(model_diff.summary())

The levels regression is shown as a comparison. The growth-rate regression is more appropriate when the level series are non-stationary.


### 3.2 Price changes and future production/consumption

I check whether oil price growth is followed by changes in production and consumption.


In [ ]:

def lagged_response(price_growth, quantity_level, horizons=(3, 6, 12)):
    rows = []
    for h in horizons:
        fwd_growth = np.log(quantity_level.shift(-h)) - np.log(quantity_level)
        df = pd.concat([price_growth, fwd_growth], axis=1).dropna()
        df.columns = ["price_growth", "fwd_growth"]
        X = sm.add_constant(df["price_growth"])
        m = sm.OLS(df["fwd_growth"], X).fit(cov_type="HAC", cov_kwds={"maxlags": h})
        rows.append({
            "horizon_months": h, "n": len(df),
            "coef": m.params["price_growth"], "p_value": m.pvalues["price_growth"],
            "ci_low": m.conf_int().loc["price_growth", 0], "ci_high": m.conf_int().loc["price_growth", 1],
            "r_squared": m.rsquared,
        })
    return pd.DataFrame(rows)

print("Does oil PRICE growth predict future PRODUCTION growth? (supply response)")
prod_response = lagged_response(d["dlog_price"], oil["oil_production_kbd"])
display(prod_response.round(4))

print("\nDoes oil PRICE growth predict future CONSUMPTION growth? (demand response)")
cons_response = lagged_response(d["dlog_price"], oil["oil_consumption_kbbl"])
display(cons_response.round(4))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, df, title, color in [
    (axes[0], prod_response, "Production response to price growth", "#2ca02c"),
    (axes[1], cons_response, "Consumption response to price growth", "#9467bd"),
]:
    ax.errorbar(df["horizon_months"], df["coef"],
                yerr=[df["coef"] - df["ci_low"], df["ci_high"] - df["coef"]],
                fmt="o-", color=color, capsize=4)
    ax.axhline(0, color="k", lw=1, ls="--")
    ax.set_xlabel("Horizon (months ahead)")
    ax.set_ylabel("Coefficient (95% CI)")
    ax.set_title(title)
    ax.set_xticks(df["horizon_months"])
plt.tight_layout()
plt.show()


The results show how the response changes over different horizons. These are associations, not causal estimates.


### 3.3 Autocorrelation

Finally, I check whether monthly returns are related to their own past values.


In [ ]:

fig, axes = plt.subplots(3, 2, figsize=(12, 8))
for i, (name, s) in enumerate(markets):
    ret = np.log(s).diff().dropna()
    plot_acf(ret, lags=24, ax=axes[i, 0], title=f"{name}: ACF of monthly log-returns")
    plot_pacf(ret, lags=24, ax=axes[i, 1], method="ywm", title=f"{name}: PACF of monthly log-returns")
plt.tight_layout()
plt.show()


The return autocorrelations are generally small, suggesting limited linear predictability in monthly returns.


## Notebook 1 summary

The first notebook describes the data and examines basic relationships. The second notebook focuses directly on large shocks and price recovery.
